In [ ]:
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

plt.style.use("dark_background")

import re
import json
import networkx as nx
from pathlib import Path
import glob
import warnings
warnings.filterwarnings("ignore")

import os
import sys
sys.path.insert(0, os.getenv("REDDIT_CODE_DIR"))
import reddit_helpers
import importlib
importlib.reload(reddit_helpers)
from reddit_helpers import *

# Load data

In [52]:
files = sorted(COMMENTS_PARQUET_DIR.glob("*.parquet")) + sorted(POSTS_PARQUET_DIR.glob("*.parquet"))

def label(p):
    tag = "comments" if p.parent == COMMENTS_PARQUET_DIR else "posts"
    return f"{p.name}"

options = [(label(p), p) for p in files]
options = sorted(options, key=lambda x: x[0].lower())

if not options:
    print("No parquet files found. Run Convert_JSONL_to_Parquet.py first.")
else:
    print(f"Found {len(options)} parquet file(s).")

Found 1852 parquet file(s).


In [53]:
SKIP_BROWSER = True
HARDCODED_SUB  = "ExPentecostal"
HARDCODED_TYPE = "posts"   # "comments" or "posts"

df = None
subreddit = None

def _load_path(path):
    global df, subreddit
    subreddit = path.name.split('_')[1]
    load = "comments" if "comments" in path.name else "posts"
    print(f"Loading {path.name} ...")
    df = load_subreddit(subreddit, load=load, clean=True, fmt='parquet', partition=1)
    print(f"Loaded: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns  |  subreddit='{subreddit}'")
    display(df.head(3))

if SKIP_BROWSER:
    _dir = COMMENTS_PARQUET_DIR if HARDCODED_TYPE == "comments" else POSTS_PARQUET_DIR
    _matches = sorted(_dir.glob(f"r_{HARDCODED_SUB}_{HARDCODED_TYPE}*.parquet"))
    if not _matches:
        raise FileNotFoundError(f"No parquet found for sub='{HARDCODED_SUB}' type='{HARDCODED_TYPE}' in {_dir}")
    _load_path(_matches[0])
else:
    dropdown = widgets.Dropdown(options=options, description="File:", layout=widgets.Layout(width="600px"))
    btn = widgets.Button(description="Load", button_style="primary")
    out = widgets.Output()

    def on_load(b):
        with out:
            clear_output()
            _load_path(dropdown.value)

    btn.on_click(on_load)
    display(widgets.VBox([dropdown, btn, out]))

Loading r_ExPentecostal_posts_1.parquet ...
Loaded: 5,557 rows × 12 columns  |  subreddit='ExPentecostal'


,id,subreddit,author,created_utc,title,selftext,score,num_comments,upvote_ratio,stickied,distinguished,over_18
0,25an74,ExPentecostal,thebedivere,2014-05-11 18:21:13,Hello and welcome!,"Hello everyone, and welcome! I created this as...",12,6,1.0,False,None,False
1,2q9cw3,ExPentecostal,thebedivere,2014-12-24 07:23:44,What is your experience with speaking in tongues?,I remember when I was 'filled with the holy gh...,3,19,1.0,False,None,False
2,3gcjmm,ExPentecostal,pagankylie27,2015-08-09 13:34:19,Pentecostal to pagan,Since my twenties I have been a pentecostal un...,2,3,1.0,False,None,False


In [54]:
vc = df.author.value_counts()
vc.head(10)

author
Frosty-Common-6205    63
crt894                62
imfinallyhere1994     47
Newdays2052           47
IamCeriella           46
RoninMs               45
Bitemebitch00         42
Familiar-Support      40
Michblanch            38
hhandhillsong         35
Name: count, dtype: int64

In [55]:
author = vc.index[10]
df_authors = df[df.author == author]

if hasattr(df_authors, "body"):
    corpus = df_authors.body.values
elif hasattr(df_authors, "selftext"):
    corpus = df_authors.selftext.values
corpus = [ re.sub(r"\s+", " ", str(text)).strip() for text in corpus ]
len(corpus)
for i in range(5):
    print(f"--- Post {i+1} ---")
    print(corpus[i][:2000])
    print()

--- Post 1 ---
From the age of 5 to 17 I went to a Pentecostal church. After turning 18 I moved and haven't set foot in a church for 4 years now. Me and my partner of almost 2 years became pregnant October of this year. I was excited but nervous and never thought I would get pregnant. Everything was going fine until I started having pain and not feeling pregnant anymore. We went to the doctor so many times those 2 months. At first they said I had a miscarriage. turns out I had a ectopic pregnancy. I had to have surgery they even had to take the tube out. Now I have only one. That was the hardest thing to go threw in my life. Alot of people that I told said maybe god did it to get my attention. Even my mother that doesn't want to see me said the same even called our unborn baby a heathen. Saying god brought evil into the world. Basically punishing me for leaving the church he killed my baby.it hurts to hear that like I did something wrong.💔 We miss our baby. 😢( My mother is still going 

# Ollama

In [56]:
import time
import ollama
import pandas as pd

_models_df = pd.read_csv(DATA_DIR / "_Created" /"model_tags_clean.csv")
_models_str = _models_df.sample(50).to_string(index=False)

models = []
for m in ollama.list().models:
    models.append(m.model)

_models_df = _models_df[_models_df["tag"].isin(models)].sort_values("size")
_models_df

,model,tag,size,updated,Image,Text,context
1635,gemma3,gemma3:1b,0.795898,365,0,1,32000
1540,gemma2,gemma2:2b,1.600000,365,0,1,8000
2800,llama3.2,llama3.2:3b-instruct-q5_K_M,2.300000,365,0,1,128000
1636,gemma3,gemma3:4b,3.300000,365,1,1,128000
2674,llama3.1,llama3.1:8b-instruct-q4_K_M,4.900000,365,0,1,128000
3157,mistral,mistral:7b-instruct-v0.3-q5_K_S,5.000000,300,0,1,32000
1665,gemma3n,gemma3n:e2b-it-q4_K_M,5.600000,270,0,1,32000
3239,mistral-nemo,mistral-nemo:12b-instruct-2407-q4_K_M,7.500000,240,0,1,1000000
2822,llama3.2-vision,llama3.2-vision:11b-instruct-q4_K_M,7.800000,330,1,1,128000
1637,gemma3,gemma3:12b,8.100000,365,1,1,128000


In [57]:
from IPython.display import Markdown

OLLAMA_MODEL = "mistral-nemo:12b-instruct-2407-q4_K_M"

THEME_SCHEMA = {
    "type": "object",
    "description": (
        "A set of distinct, non-overlapping themes found in the user's posts. "
        "Each key must be a concise but descriptive phrase (3-7 words) that names "
        "a specific recurring idea — e.g. 'Rejection of authoritarian church rules' "
        "rather than vague labels like 'Religion' or 'Church'. "
        "Themes must not duplicate or substantially overlap each other."
    ),
    "additionalProperties": {
        "type": "object",
        "properties": {
            "user_perspective_on_theme": {
                "type": "string",
                "description": (
                    "One sentence summarising the user's specific stance or feeling "
                    "on this theme, written in third person."
                )
            },
            "direct_quotes": {
                "type": "array",
                "description": "1-3 verbatim excerpts from the posts that best illustrate this theme.",
                "items": {"type": "string"},
                "minItems": 1,
                "maxItems": 3,
            }
        },
        "required": ["user_perspective_on_theme", "direct_quotes"],
        "additionalProperties": False
    }
}

def summarize_author_themes(posts, subreddit, model, max_posts, max_chars,
                             max_tokens=2048, num_ctx=4096):
    sampled = list(posts)[:max_posts]
    formatted = "\n\n---\n\n".join(
        f"[POST {i+1}]: {p[:max_chars]}{'...' if len(p) > max_chars else ''}"
        for i, p in enumerate(sampled)
    )

    system_prompt = (
        "You are a qualitative research analyst studying online religious discourse.\n"
        "You will receive a collection of Reddit comments written by a single anonymous user.\n"
        "Your task: identify and summarize the recurring themes, beliefs, and concerns in their writing.\n\n"
        "CRITICAL RULES — violating any of these is a failure:\n"
        "- These are third-party text samples. Do NOT address or speak to the author.\n"
        "- Do NOT offer emotional support, consolation, advice, or encouragement.\n"
        "- Do NOT use 'you' or 'your' when referring to the author.\n"
        "- Refer to the author in third-person only: 'the user', 'this commenter', 'the author'.\n"
        "- Return valid JSON only. The JSON must be an object where each key is a theme.\n\n"
        "THEME NAMING RULES:\n"
        "- Each theme key must be a specific, descriptive phrase (3-7 words).\n"
        "  Good: 'Discomfort with manipulative worship practices'\n"
        "  Bad:  'Church', 'Religion', 'Beliefs'\n"
        "- Themes must be mutually exclusive — no two themes should substantially overlap.\n"
        "- Prefer precise language that captures the user's actual concern, not a generic category.\n"
        "- Aim for 3-6 themes total; merge minor points rather than creating thin themes."
    )

    user_prompt = (
        f"Below is a set of {len(sampled)} comments from the r/{subreddit} Subredit,"
        f"each of which was written by the same user. Identify and summarize the recurring themes in their writing."
        f"\n\nPOSTS START BELOW\n\n"
        f"{formatted}"
    )

    response = ollama.chat(
        model=model,
        think=False,
        format=THEME_SCHEMA,
        options={
            "temperature": 0.2,
            "num_predict": max_tokens,
            "num_ctx": num_ctx,
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
    )
    return json.loads(response["message"]["content"])

def render_themes(themes):
    lines = []
    for theme, data in themes.items():
        lines.append(f"- **{theme}**")
        lines.append(f"  - *{data['user_perspective_on_theme']}*")
        for q in data["direct_quotes"]:
            lines.append(f'  - > "{q}"')
    display(Markdown("\n".join(lines)))

result = summarize_author_themes(
    corpus, subreddit, OLLAMA_MODEL, max_posts=10, max_chars=800
)
render_themes(result)

- **Trauma from Religious Experiences**
  - *The user recounts a traumatic experience of an ectopic pregnancy, which they attribute to divine intervention by some people in their life. They express distress at the insensitivity shown by others, including their mother, who suggested that the unborn baby was a 'heathen'.*
  - > "maybe god did it to get my attention"
  - > "called our unborn baby a heathen"
- **Discomfort with Manipulative Religious Practices**
  - *The user feels uncomfortable and manipulated by certain practices in the Pentecostal church they attended, such as Bible quizzing competitions, being 'drunk on the spirit', and excessive attention-seeking behavior during worship.*
  - > "They act like spoiled bratz"
  - > "Laying hands on me and shaking my head like I had no brain in there"
  - > "show off...want the attention"
- **Struggle with Religious Dress Codes**
  - *The user expresses discomfort with the dress code enforced at their church, particularly for women. They find it hypocritical that while makeup and revealing clothing are discouraged, tight-fitting skirts and dresses are worn to accentuate curves.*
  - > "Why don't they see that they are more revealing than pants"
  - > "To me they show too much curves"
- **Depression and Isolation in Religious Environments**
  - *The user describes feeling depressed while living the life of a Pentecostal, to the point where they might have considered harming themselves or others if not helped. They find that people outside the church are more helpful in their recovery than those within it.*
  - > "Depression is deadly I know from experience"
  - > "But no one wants to seem to help in the church just smile and hide it"
- **Rejection of Homophobia and Acceptance of LGBTQ+ Community**
  - *The user rejects the homophobic views they were exposed to in their religious environment. They express support for the LGBTQ+ community, encouraging them to be true to themselves despite religious opposition.*
  - > "They say that being lesbian or gay etc. That they are going to hell"
  - > "You are worth more than you know you are very special"

## Quote verification

In [58]:
import difflib

MATCH_THRESHOLD = 0.75

def quote_coverage(quote, post):
    """Fraction of quote characters found (in order) within post."""
    q, p = quote.lower(), post.lower()
    if q in p:
        return 1.0
    matched = sum(
        block.size
        for block in difflib.SequenceMatcher(None, q, p).get_matching_blocks()
    )
    return matched / len(q) if q else 0.0

rows = []
for theme, data in result.items():
    for quote in data["direct_quotes"]:
        best_ratio = max(quote_coverage(quote, post) for post in corpus)
        rows.append({
            "theme": theme,
            "quote": quote[:80] + ("..." if len(quote) > 80 else ""),
            "coverage": round(best_ratio, 3),
            "pass": best_ratio >= MATCH_THRESHOLD,
        })

df_quotes = pd.DataFrame(rows)
df_quotes

,theme,quote,coverage,pass
0,Trauma from Religious Experiences,maybe god did it to get my attention,1.000,True
1,Trauma from Religious Experiences,called our unborn baby a heathen,1.000,True
2,Discomfort with Manipulative Religious Practices,They act like spoiled bratz,1.000,True
3,Discomfort with Manipulative Religious Practices,Laying hands on me and shaking my head like I ...,1.000,True
4,Discomfort with Manipulative Religious Practices,show off...want the attention,0.759,True
5,Struggle with Religious Dress Codes,Why don't they see that they are more revealin...,1.000,True
6,Struggle with Religious Dress Codes,To me they show too much curves,0.548,False
7,Depression and Isolation in Religious Environm...,Depression is deadly I know from experience,1.000,True
8,Depression and Isolation in Religious Environm...,But no one wants to seem to help in the church...,0.957,True
9,Rejection of Homophobia and Acceptance of LGBT...,They say that being lesbian or gay etc. That t...,1.000,True
